# Step 4: PyTorch Dataset & DataLoaders Pipeline

Implements:
- PyTorch `DeepfakeAudioDataset`
- Stratified 70/15/15 Train / Validation / Test Splitting
- Batch loading and tensor conversion

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import soundfile as sf
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

BASE_DIR = Path.cwd().parent if Path.cwd().name == "src" else Path.cwd()
DATA_PROCESSED_DIR = BASE_DIR / "data" / "processed_clean"
TARGET_SAMPLES = 64000


### Dataset Class Definition

In [ ]:
class DeepfakeAudioDataset(Dataset):
    def __init__(self, df: pd.DataFrame, target_samples: int = 64000):
        self.df = df.reset_index(drop=True)
        self.target_samples = target_samples
        self.paths = self.df["proc_path"].tolist()
        self.labels = self.df["label_id"].tolist()
        self.filenames = self.df["file_name"].tolist()

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int):
        waveform, sr = sf.read(self.paths[idx], dtype="float32")
        if waveform.ndim > 1:
            waveform = np.mean(waveform, axis=1)

        if len(waveform) < self.target_samples:
            waveform = np.pad(waveform, (0, self.target_samples - len(waveform)), mode="constant")
        elif len(waveform) > self.target_samples:
            waveform = waveform[:self.target_samples]

        return {
            "input_values": torch.from_numpy(waveform).float(),
            "label": torch.tensor(self.labels[idx], dtype=torch.long),
            "filename": self.filenames[idx]
        }


In [ ]:
# Create Stratified Splits and DataLoaders
manifest_df = pd.read_csv(DATA_PROCESSED_DIR / "manifest.csv")

train_df, temp_df = train_test_split(manifest_df, test_size=0.30, random_state=42, stratify=manifest_df["label_id"])
val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=42, stratify=temp_df["label_id"])

train_loader = DataLoader(DeepfakeAudioDataset(train_df), batch_size=16, shuffle=True)
val_loader = DataLoader(DeepfakeAudioDataset(val_df), batch_size=16, shuffle=False)
test_loader = DataLoader(DeepfakeAudioDataset(test_df), batch_size=16, shuffle=False)

print(f"DataLoaders Created:")
print(f"  Train: {len(train_df):,} samples ({len(train_loader)} batches)")
print(f"  Val:   {len(val_df):,} samples ({len(val_loader)} batches)")
print(f"  Test:  {len(test_df):,} samples ({len(test_loader)} batches)")

# Inspect one batch
batch = next(iter(train_loader))
print(f"\nBatch Shape: input_values={batch['input_values'].shape}, labels={batch['label'].shape}")
